# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is FAIR-compliant, making it straightforward to explore its record sets, fields, and structure programmatically.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")
print(f"Cite as: {meta.citeAs}")
print(f"Keywords: {meta.keywords}")

## 2. Data Overview
Review available **record sets**, **fields**, columns, and their `@id`s.

The structure of Croissant datasets uses [RecordSet](https://mlcommons.org/croissant/1.0/#recordset) entities identified by `@id`. Fields (logical columns) are addressed via their `@id`s as well. Let's enumerate them programmatically.

In [ ]:
# Enumerate record sets and their fields
record_sets = list(meta.recordSets)
print(f"Found {len(record_sets)} record set(s)\n")
overview = {}
for rs in record_sets:
    print(f"RecordSet: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    field_ids = []
    fields = rs.get('fields', [])
    for f in fields:
        fid = f['@id']
        field_ids.append(fid)
        print(f"  Field: {fid} (name: {f.get('name','N/A')}, dataType: {f.get('dataType','N/A')})")
        if 'column' in f and isinstance(f['column'], dict):
            print(f"    Column: {f['column'].get('@id', 'N/A')}")
        elif 'column' in f and isinstance(f['column'], list):
            for col in f['column']:
                print(f"    Column: {col.get('@id', 'N/A')}")
    overview[rs['@id']] = field_ids
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Refer to their `@id`s as above.

In [ ]:
# We'll load data for each record set in the dataset using their @id
dataframes = {}
for rs_id in overview:
    print(f"Loading records for RecordSet: {rs_id}")
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    print(f"Loaded {df.shape[0]} records, columns: {list(df.columns)}\n")
    dataframes[rs_id] = df

# We'll show the first record set and its head
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"Columns for {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here's an example based on available numeric fields. Please reference field `@id`s above to ensure field selection.

In [ ]:
from IPython.display import display

# For demonstration, select the first numeric field from the first record set
first_rs_id = record_sets[0]['@id'] if record_sets else None
df = dataframes[first_rs_id]

# Try to find a numeric field
candidate_numeric = None
for fieldinfo in meta.recordSets[0].get('fields', []):
    if fieldinfo.get('dataType', '').lower() in ('number', 'float', 'integer', 'double'):
        candidate_numeric = fieldinfo['@id']
        break
if not candidate_numeric:
    # If not found, try to select columns with likely numeric values
    numcols = df.select_dtypes('number').columns
    candidate_numeric = numcols[0] if len(numcols) else None

if candidate_numeric is not None:
    # Remove NaN
    valid = df[candidate_numeric].dropna()
    thresh_val = valid.mean() if valid.size else 0
    filtered_df = df[df[candidate_numeric] > thresh_val]
    print(f"Filtered records with {candidate_numeric} > {thresh_val} (mean): {filtered_df.shape[0]}")

    norm_col = f"{candidate_numeric}_normalized"
    filtered_df[norm_col] = (filtered_df[candidate_numeric] - filtered_df[candidate_numeric].mean()) / filtered_df[candidate_numeric].std()
    display(filtered_df[[candidate_numeric, norm_col]].head())

    # Try to find a group field (e.g., any non-numeric categorical value)
    group_field = None
    for c in df.columns:
        if df[c].dtype == 'object' and c != candidate_numeric:
            group_field = c
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[candidate_numeric].mean()
        print(f"Grouped data by {group_field} (mean {candidate_numeric}):")
        display(grouped_df.to_frame().head())
else:
    print("No numeric field found for EDA in the first record set.")

## 5. Visualization
Visualize data distributions or relationships between fields using e.g. matplotlib or seaborn. Here, we show a histogram of the selected numeric field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_numeric is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[candidate_numeric].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {candidate_numeric}")
    plt.xlabel(candidate_numeric)
    plt.ylabel('Frequency')
    plt.show()

    # If a second numeric field, show correlation
    numcols = df.select_dtypes('number').columns
    if len(numcols) > 1:
        othercol = [col for col in numcols if col != candidate_numeric][0]
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=df, x=candidate_numeric, y=othercol)
        plt.title(f"Scatter plot: {candidate_numeric} vs {othercol}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

We demonstrated how to load and explore the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using `mlcroissant`, programmatically listing record sets, fields, and performing initial filtering and visualization. For detailed analysis, please refer to the field `@id`s and domain context in the dataset documentation.